<a href="https://colab.research.google.com/github/akim201104-creator/UTS_ZIDANE_ZAMIL_HAKIM/blob/main/UTS_BIG%20DATA_ZIDAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install google-play-scraper

In [ ]:
from google_play_scraper import reviews, Sort
import csv

result, _ = reviews(
    'com.tinder',
    lang='id',
    country='id',
    sort=Sort.NEWEST,
    count=100,
    filter_score_with=None
)

filename = 'ulasan_google_play.csv'


with open(filename, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['userName', 'score', 'at', 'content'])
    writer.writeheader()
    for review in result:

        writer.writerow({
            'userName': review['userName'],
            'score': review['score'],
            'at': review['at'],
            'content': review['content']
        })

print(f"Berhasil menyimpan {len(result)} ulasan ke '{filename}'")

In [ ]:
pip install transformers

In [ ]:
from transformers import pipeline
import pandas as pd

# Load the previously saved reviews from the CSV file into a DataFrame
df = pd.read_csv(filename)

### Sentiment Analysis using `w11wo/indonesian-roberta-base-prdect-id`

Now, let's load the specified model and apply it to the `content` column of our DataFrame. This will classify each review.

In [ ]:
classifier = pipeline("sentiment-analysis", model="w11wo/indonesian-roberta-base-prdect-id")

# Apply the classifier to the content of the reviews
# Handle potential NaN values by converting them to empty strings or filtering
df['predicted_sentiment'] = df['content'].fillna('').apply(lambda x: classifier(x)[0]['label'] if x.strip() != '' else 'UNKNOWN')
df['sentiment_score'] = df['content'].fillna('').apply(lambda x: classifier(x)[0]['score'] if x.strip() != '' else 0.0)

### Displaying Analysis Results

Here are the first few rows of your DataFrame with the added sentiment analysis results:

In [ ]:
display(df.head())

### Sentiment Distribution

Let's also look at the overall distribution of the predicted sentiments.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sentiment_counts = df['predicted_sentiment'].value_counts().reset_index()
sentiment_counts.columns = ['Sentiment', 'Count']

plt.figure(figsize=(8, 6))
sns.barplot(x='Sentiment', y='Count', data=sentiment_counts, hue='Sentiment', palette='viridis', legend=False)
plt.title('Distribution of Predicted Sentiments')
plt.xlabel('Sentiment')
plt.ylabel('Number of Reviews')
plt.show()